# Phase 3: Transformer Fine-Tuning

This notebook reviews the DistilBERT complaint-classification run. The training pipeline uses `text_transformer`, not `text_ml_clean`, because transformer models prefer relatively natural text.

Pipeline:

Complaint narrative -> Tokenizer -> DistilBERT -> Classification head -> Product category


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
REPORTS = PROJECT_ROOT / 'artifacts' / 'reports'
MODEL_DIR = PROJECT_ROOT / 'artifacts' / 'models' / 'distilbert_complaint_classifier'


## Transformer Results

In [ ]:
transformer_results = pd.read_csv(REPORTS / 'transformer_model_results.csv')
transformer_results


## Baseline vs Transformer

In [ ]:
comparison = pd.read_csv(REPORTS / 'baseline_vs_transformer.csv')
comparison


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=comparison, x='macro_f1', y='model_name', hue='stage')
plt.title('Macro F1: Baseline vs Transformer')
plt.xlabel('Macro F1')
plt.ylabel('Model')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


## Confusion Matrix

In [ ]:
cm = pd.read_csv(REPORTS / 'transformer_confusion_matrix.csv', index_col=0)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=False)
plt.title('DistilBERT Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


## Error Analysis

In [ ]:
errors = pd.read_csv(REPORTS / 'transformer_error_examples.csv')
errors.shape


In [ ]:
errors.groupby(['actual', 'predicted']).size().sort_values(ascending=False).head(15)


In [ ]:
errors.sample(min(10, len(errors)), random_state=42)


## Label Mapping

In [ ]:
label_mapping = json.loads((MODEL_DIR / 'label_mapping.json').read_text())
label_mapping


## Run Commands\n\nDistilBERT pilot:\n\n```bash\npython -m src.pipelines.phase3_finetune_transformer --max-samples-per-class 1000 --epochs 1 --train-batch-size 8 --eval-batch-size 16\n```\n\nBERT pilot comparison:\n\n```bash\npython -m src.pipelines.phase3_finetune_transformer --model-name bert-base-uncased --max-samples-per-class 1000 --epochs 1 --train-batch-size 4 --eval-batch-size 8\n```\n\nFull 90k-row DistilBERT run:\n\n```bash\npython -m src.pipelines.phase3_finetune_transformer --full-dataset --epochs 2 --train-batch-size 8 --eval-batch-size 16\n```\n